In [8]:
import os, shutil
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("java.exe found at:", shutil.which("java"))

JAVA_HOME: C:\Java\jdk-17
java.exe found at: C:\Java\jdk-17\bin\java.EXE


In [9]:
import pyspark, os
pyspark_home = os.path.dirname(pyspark.__file__)
print("PySpark home:", pyspark_home)
bin_dir = os.path.join(pyspark_home, "bin")
print("bin dir exists:", os.path.exists(bin_dir))
if os.path.exists(bin_dir):
    print(os.listdir(bin_dir))
print("SPARK_HOME env:", os.environ.get("SPARK_HOME"))

PySpark home: c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark
bin dir exists: True
['beeline', 'beeline.cmd', 'docker-image-tool.sh', 'find-spark-home', 'find-spark-home.cmd', 'load-spark-env.cmd', 'load-spark-env.sh', 'pyspark', 'pyspark.cmd', 'pyspark2.cmd', 'run-example', 'run-example.cmd', 'spark-class', 'spark-class.cmd', 'spark-class2.cmd', 'spark-connect-shell', 'spark-shell', 'spark-shell.cmd', 'spark-shell2.cmd', 'spark-sql', 'spark-sql.cmd', 'spark-sql2.cmd', 'spark-submit', 'spark-submit.cmd', 'spark-submit2.cmd', 'sparkR', 'sparkR.cmd', 'sparkR2.cmd']
SPARK_HOME env: None


In [10]:
import os
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

SPARK_HOME: None


In [11]:
import os

# --- Sanity check: these MUST be visible here, not just in cmd ---
print("JAVA_HOME  :", os.environ.get("JAVA_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

assert os.environ.get("JAVA_HOME"), "JAVA_HOME not set — restart VS Code fully after setx, not just the terminal"
assert os.environ.get("HADOOP_HOME"), "HADOOP_HOME not set — restart VS Code fully after setx, not just the terminal"

winutils_path = os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")
assert os.path.exists(winutils_path), f"winutils.exe not found at {winutils_path}"
print("winutils found at:", winutils_path)

JAVA_HOME  : C:\Java\jdk-17
HADOOP_HOME: C:\hadoop
winutils found at: C:\hadoop\bin\winutils.exe


In [12]:
# --- Base project directory (change here ONLY if you move the project) ---
BASE_DIR = r"C:\covid_pipeline"

BRONZE_DIR = os.path.join(BASE_DIR, "bronze")
SILVER_DIR = os.path.join(BASE_DIR, "silver")
GOLD_DIR   = os.path.join(BASE_DIR, "gold")

for d in [BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    os.makedirs(d, exist_ok=True)
    print("Ready:", d)

# --- Raw source files live directly inside the bronze folder (as-uploaded, unmodified) ---
RAW_VACCINATION = os.path.join(BRONZE_DIR, "country_vaccinations.csv")
RAW_CASES_DEATHS = os.path.join(BRONZE_DIR, "covid19_cases_deaths.csv")
RAW_POPULATION   = os.path.join(BRONZE_DIR, "world_population.csv")
RAW_OXCGRT       = os.path.join(BRONZE_DIR, "oxcgrt_stringency_index.csv")
RAW_WORLDBANK    = os.path.join(BRONZE_DIR, "world_bank_income_classification.csv")

# --- Silver output paths (Parquet) ---
SILVER_VACCINATION = os.path.join(SILVER_DIR, "vaccination")
SILVER_CASES_DEATHS = os.path.join(SILVER_DIR, "cases_deaths")
SILVER_POPULATION   = os.path.join(SILVER_DIR, "population")
SILVER_OXCGRT       = os.path.join(SILVER_DIR, "oxcgrt")
SILVER_WORLDBANK    = os.path.join(SILVER_DIR, "worldbank")

# --- Gold output path ---
GOLD_INTEGRATED = os.path.join(GOLD_DIR, "gold_integrated")

Ready: C:\covid_pipeline\bronze
Ready: C:\covid_pipeline\silver
Ready: C:\covid_pipeline\gold


In [13]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("covid_vaccination_disparity_pipeline")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")   # small dataset -> fewer partitions, faster locally
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.io.native.lib.available", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark master :", spark.sparkContext.master)

Spark version: 3.5.3
Spark master : local[*]


In [14]:
# --- Final smoke test: confirm Spark can actually WRITE parquet locally (this is the step that used to fail) ---
test_df = spark.range(5)
test_path = os.path.join(BASE_DIR, "_smoke_test")
test_df.write.mode("overwrite").parquet(test_path)
print("Parquet write test succeeded at:", test_path)
spark.read.parquet(test_path).show()

Parquet write test succeeded at: C:\covid_pipeline\_smoke_test
+---+
| id|
+---+
|  2|
|  3|
|  4|
|  0|
|  1|
+---+

